# *初始化操作*

resnet18_cifar10为初始化的模型框架，后续可以考虑在其中加入注意力机制一类的层

model_to_quantize_model的目的是将模型转换为可以量化的模型，缺少这一步，后续使用可以量化的框架将出现问题;
<font color="Red">*注意！！！*</font>
resnet系列量化激活值会出现问题

dummy_input的目的，猜测为找出模型框架，配合visualize实现可视化

In [1]:
import sys
sys.path.append('..')
from sanity_check.backends.resnet20_cifar10 import resnet20_cifar10
from only_train_once.quantization.quant_model import model_to_quantize_model
from only_train_once import OTO
import torch

model = resnet20_cifar10()
model = model_to_quantize_model(model)
dummy_input = torch.rand(1, 3, 32, 32)
oto = OTO(model=model.cuda(), dummy_input=dummy_input.cuda())

2025-08-19 16:38:30,747 - only_train_once.quantization.quant_model - INFO - Converted 22 layers to quantized versions
E:\Huarun\TinyML_projects\geta-main\geta-main\only_train_once\quantization\quant_layers.py:334: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  weight_clip_val = torch.tensor(self.weight_clip_val, device=weight.device)
E:\Huarun\TinyML_projects\geta-main\geta-main\only_train_once\quantization\quant_layers.py:335: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  q_s =

OTO graph constructor
graph build
NodePattern mul None
NodePattern transpose None
NodePattern matmul None
Post-processing of graph completed.
Graph has 75 nodes and 83 edges.


创建可视化结构图，这里需要graphviz支持，没有也不影响后续训练

In [ ]:
# A ResNet_zig.gv.pdf will be generated to display the depandancy graph.
oto.visualize(view=False, out_dir='../cache')

# *创建数据集*

根据cifar10创建数据集，后续需要单独继承类实现其他数据集的创建

In [2]:
# from model_test import get_mstar_loaders
# 
# trainloader, testloader = get_mstar_loaders(root='../datasets/MSTAR', batch_size = 8, num_workers = 2)
from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms

trainset = CIFAR10(root='../datasets/cifar10', train=True, download=True, transform=transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, 4),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]))
testset = CIFAR10(root='../datasets/cifar10', train=False, download=True, transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]))

trainloader =  torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=4)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=4)
# for X, y in trainloader:
#     X = X
#     y = y

Files already downloaded and verified
Files already downloaded and verified


# *初始化优化器*

*重点！*

*参数说明：*
- **variant**：用于训练基准完整模型的优化器。目前支持sgd、adam和adamw。
- **lr**：初始学习率。
- **weight_decay**：标准深度神经网络（DNN）优化中的权重衰减。
- **target_group_sparsity**：目标组稀疏度，通常组稀疏度越高，意味着计算量（FLOPs）和模型大小的减少越多，同时可能会使模型性能下降得更明显。
- **start_projection_step**：开始进行位宽投射的步数。
- **projection_steps**：在开始投射步骤之后，完成位宽投射（达到目标位宽）所需的步数。
- **projection_periods**：在投射周期内，均匀地逐步实现组稀疏度。
- **start_pruning_step**：开始进行剪枝的步数。
- **pruning_steps**：在开始剪枝步骤之后，完成剪枝（达到目标组稀疏度）所需的步数。
- **pruning_periods**：在剪枝周期内，均匀地逐步实现组稀疏度。
- **bit reduction**：每个投射周期结束后最大位宽（max_bit）的减少量。
- **[min_bit,max_bit]**：初始位宽区间。最大位宽（max_bit）会在每个投射周期后按位宽减少量（bit reduction）进行缩减。

In [3]:
# optimizer = oto.hesso(
#         variant='sgd',
#         lr=0.1,
#         weight_decay=1e-4,
#         target_group_sparsity=0.1,
#         start_pruning_step=10 * len(trainloader),
#         pruning_periods=10,
#         pruning_steps=10 * len(trainloader)
#     )
sparsity = 0.7
start_projection_epoch=10
start_pruning_epoch=15
projection_epochs=50
pruning_epochs=50
optimizer = oto.geta(
    variant="adam",
    lr=1e-3,
    lr_quant=1e-3,
    first_momentum=0.9,
    weight_decay=1e-4,
    target_group_sparsity=sparsity,
    start_projection_step=start_projection_epoch * len(trainloader),
    projection_periods=10,
    projection_steps=projection_epochs * len(trainloader),
    start_pruning_step=start_pruning_epoch * len(trainloader),
    pruning_periods=10,
    pruning_steps=pruning_epochs * len(trainloader),
    bit_reduction=2,
    min_bit_wt=4,
    max_bit_wt=16,
)

2025-08-19 16:38:46,362 - GETA - INFO - Setup GETA
2025-08-19 16:38:46,364 - GETA - INFO - importance_score_criteria: {'magnitude': 0.2, 'avg_magnitude': 0.2, 'cosine_similarity': 0.2, 'taylor_first_order': 0.2, 'taylor_second_order': 0.2}
2025-08-19 16:38:46,365 - GETA - INFO - start_projection_step: 7820
2025-08-19 16:38:46,365 - GETA - INFO - projection_steps: 39100
2025-08-19 16:38:46,366 - GETA - INFO - projection_periods: 10
2025-08-19 16:38:46,366 - GETA - INFO - start_pruning_step: 11730
2025-08-19 16:38:46,368 - GETA - INFO - pruning_steps: 39100
2025-08-19 16:38:46,368 - GETA - INFO - pruning_periods: 10
2025-08-19 16:38:46,369 - GETA - INFO - pruning_period_duration: 3910
2025-08-19 16:38:46,374 - GETA - INFO - Target redundant groups per period: [31, 31, 31, 31, 31, 31, 31, 31, 31, 34]


# *损失实现*

在此处实现损失函数

In [4]:
from loss_test import TotalLoss
total_loss = TotalLoss(start_projection_epoch=start_projection_epoch,
start_pruning_epoch=start_pruning_epoch,
projection_epochs=projection_epochs,
pruning_epochs=pruning_epochs)

# *训练*

考虑训练次数和损失函数的设置

In [5]:
from tutorials.utils.utils import check_accuracy

max_epoch = 200
model.cuda()

# Every 50 epochs, decay lr by 10.0
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.7)

for epoch in range(max_epoch):
    f_avg_val = 0.0
    celoss_avg_val = 0.0
    rcrloss_avg_val = 0.0
    conloss_avg_val = 0.0
    model.train()
    lr_scheduler.step()
    for X, y in trainloader:
        X = X.cuda()
        y = y.cuda()
        y_pred,feature = model.forward(X,feature_need=True)
        celoss, rcrloss, conloss = total_loss.total_loss(y_pred=y_pred, y=y, model=model, feature=feature, epoch=epoch, RCR=True, CON=True)
        f = celoss + rcrloss + conloss
        optimizer.zero_grad()
        f.backward()
        f_avg_val += f
        celoss_avg_val += celoss
        rcrloss_avg_val += rcrloss
        conloss_avg_val += conloss
        optimizer.step()

    opt_metrics = optimizer.compute_metrics()
    accuracy1, accuracy5 = check_accuracy(model, testloader)
    f_avg_val = f_avg_val / len(trainloader)
    celoss_avg_val = celoss_avg_val / len(trainloader)
    rcrloss_avg_val = rcrloss_avg_val / len(trainloader)
    conloss_avg_val = conloss_avg_val / len(trainloader)

    print("Ep: {ep}, celoss: {celoss:.4f}, rcrloss: {rcrloss:.4f}, conloss: {conloss:.4f}, norm_all:{param_norm:.2f}, grp_sparsity: {gs:.2f}, acc1: {acc1:.4f}, norm_import: {norm_import:.2f}, norm_redund: {norm_redund:.2f}, num_grp_import: {num_grps_import}, num_grp_redund: {num_grps_redund}\n"\
         .format(ep=epoch, celoss=celoss_avg_val, rcrloss=rcrloss_avg_val, conloss=conloss_avg_val,param_norm=opt_metrics.norm_params, gs=opt_metrics.group_sparsity, acc1=accuracy1,\
         norm_import=opt_metrics.norm_important_groups, norm_redund=opt_metrics.norm_redundant_groups, \
         num_grps_import=opt_metrics.num_important_groups, num_grps_redund=opt_metrics.num_redundant_groups
        ))
    # _, s1, _ = torch.svd(model.conv1.weight, compute_uv=False)
    # _, s2, _ = torch.svd(model.linear.weight, compute_uv=False)
    # print(s1, s2)
    # print("\n")
    if (epoch + 1) % 10 == 0:  # 使用epoch+1是为了在第10、20...个epoch结束时保存
        torch.save(model, f"resnet56_epoch_{epoch}_sparsity_{str(sparsity)}.pt")

D:\softwares\Anaconda\envs\d2l\lib\site-packages\torch\optim\lr_scheduler.py:136: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


KeyboardInterrupt: 

# *结构获取*

获取整体网络和生成子网

In [ ]:
# By default OTO will construct subnet by the last checkpoint. If intermedia ckpt reaches the best performance,
# need to reinitialize OTO instance
# ckpt_path = "cache/ResNet_full_group_sparse.pt"
# oto = OTO(torch.load(ckpt_path).cuda(), dummy_input.cuda())
# then construct subnetwork
oto.construct_subnet(out_dir='./cache')

# *参数量对比*

对比压缩前后参数量

In [ ]:
import os

full_model_size = os.stat(oto.full_group_sparse_model_path)
compressed_model_size = os.stat(oto.compressed_model_path)
print("Size of full model     : ", full_model_size.st_size / (1024 ** 2), "MBs")
print("Size of compress model : ", compressed_model_size.st_size / (1024 ** 2), "MBs")

# *检查准确率*

此处两个模型准确率应当一致，full_model保存了裁剪和量化前的参数，只是没激活。

In [ ]:
full_model = torch.load(oto.full_group_sparse_model_path)
compressed_model = torch.load(oto.compressed_model_path)

acc1_full, acc5_full = check_accuracy(full_model, testloader)
print("Full model: Acc 1: {acc1}, Acc 5: {acc5}".format(acc1=acc1_full, acc5=acc5_full))

acc1_compressed, acc5_compressed = check_accuracy(compressed_model, testloader)
print("Compressed model: Acc 1: {acc1}, Acc 5: {acc5}".format(acc1=acc1_compressed, acc5=acc5_compressed))

权重信息量控制：带有0.01正则化：73.96，带有0.1正则化：69.42，带有0.001正则化：73.0，不带正则化72.84

标签熵控制：带有0.01正则化：73.4，